# UCI Regression Benchmarks

Reproduction of (a subset of) Table 1 of Lakshminarayanan et al. (2017).
We compare deep ensembles to the reference numbers reported in the
paper for PBP and MC-dropout on three UCI regression datasets:
Boston Housing, Energy Efficiency, and Yacht Hydrodynamics.

Protocol (from Hernandez-Lobato and Adams, 2015): 20 random 90/10
splits per dataset, MLP with 1 hidden layer of 50 units, 40 epochs,
batch size 100, Adam. Inputs and targets are standardized using
training-set statistics; metrics are reported on the original scale.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from urllib.request import urlretrieve
from pathlib import Path

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Datasets

We use the dataset files maintained by Yarin Gal in his MC-dropout
repository, which are the same files used by the original PBP paper
and several follow-ups.

In [ ]:
DATA_DIR = Path("data/uci")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "boston": "https://raw.githubusercontent.com/yaringal/DropoutUncertaintyExps/master/UCI_Datasets/bostonHousing/data/data.txt",
    "energy": "https://raw.githubusercontent.com/yaringal/DropoutUncertaintyExps/master/UCI_Datasets/energy/data/data.txt",
    "yacht":  "https://raw.githubusercontent.com/yaringal/DropoutUncertaintyExps/master/UCI_Datasets/yacht/data/data.txt",
}

# number of target columns; the rest are features
N_TARGETS = {"boston": 1, "energy": 2, "yacht": 1}

for name, url in DATASETS.items():
    path = DATA_DIR / f"{name}.txt"
    if not path.exists():
        print(f"Downloading {name}...")
        urlretrieve(url, path)
    else:
        print(f"{name} already present.")

In [ ]:
def load_dataset(name):
    raw = np.loadtxt(DATA_DIR / f"{name}.txt")
    n_target = N_TARGETS[name]
    if n_target == 1:
        X, y = raw[:, :-1], raw[:, -1]
    else:
        # keep only the first target (heating load for energy)
        X, y = raw[:, :-n_target], raw[:, -n_target]
    return X.astype(np.float32), y.astype(np.float32)


for name in DATASETS:
    X, y = load_dataset(name)
    print(f"{name:8s}  N={X.shape[0]:5d}  D={X.shape[1]:2d}")

## Model

`GaussianMLP` outputs `(mu, var)` with softplus on the variance head.
Same architecture as the toy regression, with input dimension matching
the dataset.

In [ ]:
class GaussianMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=50, min_var=1e-6):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, 1)
        self.fc_var = nn.Linear(hidden_dim, 1)
        self.min_var = min_var

    def forward(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        var = F.softplus(self.fc_var(h)) + self.min_var
        return mu.squeeze(-1), var.squeeze(-1)


def gaussian_nll(mu, var, y):
    return (0.5 * torch.log(var) + 0.5 * (y - mu) ** 2 / var).mean()

## Training and evaluation

For one split: standardize using training stats, train an ensemble of
M networks, predict on the test set, combine as a mixture of Gaussians,
denormalize, and report RMSE and NLL on the original scale.

Denormalization of NLL: if `y_norm = (y - mean) / std`, then a Gaussian
prediction `(mu_norm, var_norm)` on the normalized scale corresponds to
`(mu_norm * std + mean, var_norm * std^2)` on the original scale, and
`NLL_orig = NLL_norm + log(std)`.

In [ ]:
def train_one_network(X_train, y_train, n_epochs=40, batch_size=100, lr=0.01):
    in_dim = X_train.shape[1]
    model = GaussianMLP(in_dim=in_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    X_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    n = X_t.shape[0]

    for _ in range(n_epochs):
        idx = torch.randperm(n)
        for i in range(0, n, batch_size):
            b = idx[i:i + batch_size]
            opt.zero_grad()
            mu, var = model(X_t[b])
            loss = gaussian_nll(mu, var, y_t[b])
            loss.backward()
            opt.step()

    return model

In [ ]:
@torch.no_grad()
def predict_ensemble(models, X):
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    mus, varz = [], []
    for model in models:
        model.eval()
        mu, var = model(X_t)
        mus.append(mu.cpu().numpy())
        varz.append(var.cpu().numpy())
    mus = np.stack(mus)
    varz = np.stack(varz)

    mu_star = mus.mean(axis=0)
    var_star = (varz + mus ** 2).mean(axis=0) - mu_star ** 2
    return mu_star, var_star


def evaluate_split(X, y, train_idx, test_idx, split_seed, M=5):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # standardize with training stats
    X_mean, X_std = X_train.mean(0), X_train.std(0) + 1e-8
    y_mean, y_std = y_train.mean(), y_train.std() + 1e-8

    X_train_n = (X_train - X_mean) / X_std
    X_test_n  = (X_test  - X_mean) / X_std
    y_train_n = (y_train - y_mean) / y_std

    # train ensemble; seed depends on the split for more diversity
    models = []
    for m in range(M):
        torch.manual_seed(split_seed * 1000 + m)
        models.append(train_one_network(X_train_n, y_train_n))

    mu_n, var_n = predict_ensemble(models, X_test_n)

    # denormalize
    mu = mu_n * y_std + y_mean
    var = var_n * (y_std ** 2)

    # metrics on the original scale
    rmse = float(np.sqrt(np.mean((y_test - mu) ** 2)))
    nll = float(np.mean(0.5 * np.log(2 * np.pi * var) + 0.5 * (y_test - mu) ** 2 / var))

    return rmse, nll

## Sanity check on one split

Before running everything, train and evaluate on a single Boston
split to make sure the pipeline works.

In [ ]:
X, y = load_dataset("boston")
n = X.shape[0]
rng = np.random.default_rng(0)
idx = rng.permutation(n)
n_train = int(0.9 * n)
train_idx, test_idx = idx[:n_train], idx[n_train:]

rmse, nll = evaluate_split(X, y, train_idx, test_idx, split_seed=0, M=5)
print(f"Boston, one split: RMSE={rmse:.2f}  NLL={nll:.2f}")
print(f"Paper reference:   RMSE=3.28      NLL=2.41")

## Full evaluation: 20 splits per dataset

Each split takes a few seconds. Total runtime is around 5-10 minutes
on CPU for the three datasets.

In [ ]:
N_SPLITS = 20
M = 5
results = {}

for name in DATASETS:
    X, y = load_dataset(name)
    n = X.shape[0]
    n_train = int(0.9 * n)

    rmses, nlls = [], []
    for s in range(N_SPLITS):
        rng = np.random.default_rng(s)
        idx = rng.permutation(n)
        train_idx, test_idx = idx[:n_train], idx[n_train:]
        rmse, nll = evaluate_split(X, y, train_idx, test_idx, split_seed=s, M=M)
        rmses.append(rmse)
        nlls.append(nll)

    rmses = np.array(rmses)
    nlls = np.array(nlls)
    results[name] = {
        "rmse_mean": rmses.mean(), "rmse_std": rmses.std(),
        "nll_mean":  nlls.mean(),  "nll_std":  nlls.std(),
    }
    print(f"{name:8s}  RMSE={rmses.mean():.2f} +/- {rmses.std():.2f}   "
          f"NLL={nlls.mean():.2f} +/- {nlls.std():.2f}")

## Results

Comparison with the reference numbers reported in the paper.

In [ ]:
PAPER = {
    "boston": {"rmse_pbp": 3.01, "rmse_mcd": 2.97, "rmse_de": 3.28,
               "nll_pbp":  2.57, "nll_mcd":  2.46, "nll_de":  2.41},
    "energy": {"rmse_pbp": 1.80, "rmse_mcd": 1.66, "rmse_de": 2.09,
               "nll_pbp":  2.04, "nll_mcd":  1.99, "nll_de":  1.38},
    "yacht":  {"rmse_pbp": 1.02, "rmse_mcd": 1.11, "rmse_de": 1.58,
               "nll_pbp":  1.63, "nll_mcd":  1.55, "nll_de":  1.18},
}

rows = []
for name in DATASETS:
    r = results[name]
    p = PAPER[name]
    rows.append({
        "Dataset": name,
        "RMSE (PBP)":        f"{p['rmse_pbp']:.2f}",
        "RMSE (MC-dropout)": f"{p['rmse_mcd']:.2f}",
        "RMSE (DE paper)":   f"{p['rmse_de']:.2f}",
        "RMSE (ours)":       f"{r['rmse_mean']:.2f} +/- {r['rmse_std']:.2f}",
        "NLL (PBP)":         f"{p['nll_pbp']:.2f}",
        "NLL (MC-dropout)":  f"{p['nll_mcd']:.2f}",
        "NLL (DE paper)":    f"{p['nll_de']:.2f}",
        "NLL (ours)":        f"{r['nll_mean']:.2f} +/- {r['nll_std']:.2f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
os.makedirs("figures", exist_ok=True)
df.to_csv("figures/uci_results.csv", index=False)
print("Saved to figures/uci_results.csv")